In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from functions import (load_month, load_month_sa, load_month_pen, mapd_clean_merge)
!pip install openpyxl

# Settings
monthlist = [f"{m:02d}" for m in range(1, 13)] 
y = 2013


### Plan Enrollment and Contract Data

In [2]:
# Load and combine monthly data
plan_data = pd.concat([load_month(m, y) for m in monthlist], ignore_index=True)

# Sort data
plan_data = plan_data.sort_values(['contractid', 'planid', 'state', 'county', 'month'])

# Fill fips by state/county groups
plan_data['fips'] = plan_data.groupby(['state', 'county'])['fips'].transform(
    lambda x: x.ffill().bfill()
)

# Fill plan-level attributes
plan_cols = ['plan_type', 'partd', 'snp', 'egbp', 'plan_name']
for col in plan_cols:
    plan_data[col] = plan_data.groupby(['contractid', 'planid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Fill contract-level attributes
contract_cols = ['org_type', 'org_name', 'org_marketing_name', 'parent_org']
for col in contract_cols:
    plan_data[col] = plan_data.groupby(['contractid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Sort for aggregation
plan_data = plan_data.sort_values(['contractid', 'planid', 'fips', 'year', 'month'])

# Aggregate to yearly level
def agg_plan_year(group):
    nonmiss = group['enrollment'].notna()
    n = nonmiss.sum()
    enroll_vals = group.loc[nonmiss, 'enrollment']
    
    return pd.Series({
        'n_nonmiss': n,
        'avg_enrollment': enroll_vals.mean() if n > 0 else np.nan,
        'sd_enrollment': enroll_vals.std() if n > 1 else np.nan,
        'min_enrollment': enroll_vals.min() if n > 0 else np.nan,
        'max_enrollment': enroll_vals.max() if n > 0 else np.nan,
        'first_enrollment': enroll_vals.iloc[0] if n > 0 else np.nan,
        'last_enrollment': enroll_vals.iloc[-1] if n > 0 else np.nan,
        'state': group['state'].iloc[-1],
        'county': group['county'].iloc[-1],
        'org_type': group['org_type'].iloc[-1],
        'plan_type': group['plan_type'].iloc[-1],
        'partd': group['partd'].iloc[-1],
        'snp': group['snp'].iloc[-1],
        'egbp': group['egbp'].iloc[-1],
        'org_name': group['org_name'].iloc[-1],
        'org_marketing_name': group['org_marketing_name'].iloc[-1],
        'plan_name': group['plan_name'].iloc[-1],
        'parent_org': group['parent_org'].iloc[-1],
        'contract_date': group['contract_date'].iloc[-1]
    })

plan_data_2013 = plan_data.groupby(['contractid', 'planid', 'fips', 'year']).apply(agg_plan_year).reset_index()
print(f"Plan data shape: {plan_data_2013.shape}")

/tmp/ipykernel_1182301/627019427.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()
/tmp/ipykernel_1182301/627019427.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()


Plan data shape: (2222251, 23)


/tmp/ipykernel_1182301/627019427.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  plan_data_2013 = plan_data.groupby(['contractid', 'planid', 'fips', 'year']).apply(agg_plan_year).reset_index()


### Service Area Data

In [3]:
# Load and combine monthly service area data
service_year = pd.concat([load_month_sa(m, y) for m in monthlist], ignore_index=True)

# Sort for stable fills
service_year = service_year.sort_values(['contractid', 'fips', 'state', 'county', 'month'])

# Fill fips by state/county groups
service_year['fips'] = service_year.groupby(['state', 'county'])['fips'].transform(
    lambda x: x.ffill().bfill()
)

# Fill contract-level attributes
contract_cols_sa = ['plan_type', 'partial', 'eghp', 'org_type', 'org_name']
for col in contract_cols_sa:
    service_year[col] = service_year.groupby(['contractid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Collapse to yearly: one row per contract x county (fips) x year
service_year = service_year.sort_values(['contractid', 'fips', 'year', 'month'])

service_data_2013 = service_year.groupby(['contractid', 'fips', 'year']).agg({
    'state': 'last',
    'county': 'last',
    'org_name': 'last',
    'org_type': 'last',
    'plan_type': 'last',
    'partial': 'last',
    'eghp': 'last',
    'ssa': 'last',
    'notes': 'last'
}).reset_index()

print(f"Service area data shape: {service_data_2013.shape}")

/tmp/ipykernel_1182301/1777593515.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()
/tmp/ipykernel_1182301/1777593515.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()


Service area data shape: (376589, 12)


# Premium Data

In [4]:
# MA landscape data (A to M)
ma_path_a = "../ma-data/ma/landscape/Extracted Data/2013LandscapeSource file MA_AtoM 11212012.csv"
ma_col_names = ["state","county","org_name","plan_name","plan_type","premium","partd_deductible",
                "drug_type","gap_coverage","drug_type_detail","contractid",
                "planid","segmentid","moop","star_rating"]

ma_data_a = pd.read_csv(ma_path_a, skiprows=6, names=ma_col_names, dtype=str, encoding = "latin-1")
ma_data_a['premium'] = ma_data_a['premium'].str.replace('-', '0')
ma_data_a['partd_deductible'] = ma_data_a['partd_deductible'].str.replace('-', '0')
ma_data_a['premium'] = pd.to_numeric(ma_data_a['premium'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
ma_data_a['partd_deductible'] = pd.to_numeric(ma_data_a['partd_deductible'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
ma_data_a['planid'] = pd.to_numeric(ma_data_a['planid'], errors='coerce')
ma_data_a['segmentid'] = pd.to_numeric(ma_data_a['segmentid'], errors='coerce')

# MA landscape data (N to W)
ma_path_b = "../ma-data/ma/landscape/Extracted Data/2013LandscapeSource file MA_NtoW 11212012.csv"
ma_data_b = pd.read_csv(ma_path_b, skiprows=6, names=ma_col_names, dtype=str, encoding = "latin-1")
ma_data_b['premium'] = ma_data_b['premium'].str.replace('-', '0')
ma_data_b['partd_deductible'] = ma_data_b['partd_deductible'].str.replace('-', '0')
ma_data_b['premium'] = pd.to_numeric(ma_data_b['premium'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
ma_data_b['partd_deductible'] = pd.to_numeric(ma_data_b['partd_deductible'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
ma_data_b['planid'] = pd.to_numeric(ma_data_b['planid'], errors='coerce')
ma_data_b['segmentid'] = pd.to_numeric(ma_data_b['segmentid'], errors='coerce')

ma_data = pd.concat([ma_data_a, ma_data_b], ignore_index=True)

# MA-PD landscape data
mapd_path = "../ma-data/ma/landscape/Extracted Data/PartCD/2013/Medicare Part D 2013 Plan Report 04252013v1.xls"
mapd_col_names = ["state","county","org_name","plan_name","contractid","planid","segmentid",
                  "org_type","plan_type","snp","snp_type","benefit_type","below_benchmark",
                  "national_pdp","premium_partc",
                  "premium_partd_basic","premium_partd_supp","premium_partd_total",
                  "partd_assist_full","partd_assist_75","partd_assist_50","partd_assist_25",
                  "partd_deductible","deductible_exclusions","increase_coverage_limit",
                  "gap_coverage","gap_coverage_type"]

mapd_data_a = pd.read_excel(mapd_path, sheet_name="Alabama to Montana", skiprows=4, 
                            nrows=20936, names=mapd_col_names)
mapd_data_b = pd.read_excel(mapd_path, sheet_name="Nebraska to Wyoming", skiprows=4,
                            nrows=23808, names=mapd_col_names)
mapd_data = pd.concat([mapd_data_a, mapd_data_b], ignore_index=True)

# Clean and merge
landscape_2013 = mapd_clean_merge(ma_data, mapd_data, y)
print(f"Landscape data shape: {landscape_2013.shape}")

Landscape data shape: (56976, 11)


### Penetration Data

In [5]:
# Load and combine monthly penetration data
ma_penetration = pd.concat([load_month_pen(m, y) for m in monthlist], ignore_index=True)

# Sort and fill fips
ma_penetration = ma_penetration.sort_values(['state', 'county', 'month'])
ma_penetration['fips'] = ma_penetration.groupby(['state', 'county'])['fips'].transform(
    lambda x: x.ffill().bfill()
)

# Collapse to yearly
def agg_penetration(group):
    n_elig = group['eligibles'].notna().sum()
    n_enrol = group['enrolled'].notna().sum()
    elig_vals = group['eligibles'].dropna()
    enrol_vals = group['enrolled'].dropna()
    
    return pd.Series({
        'n_elig': n_elig,
        'n_enrol': n_enrol,
        'avg_eligibles': elig_vals.mean() if n_elig > 0 else np.nan,
        'sd_eligibles': elig_vals.std() if n_elig > 1 else np.nan,
        'min_eligibles': elig_vals.min() if n_elig > 0 else np.nan,
        'max_eligibles': elig_vals.max() if n_elig > 0 else np.nan,
        'first_eligibles': elig_vals.iloc[0] if n_elig > 0 else np.nan,
        'last_eligibles': elig_vals.iloc[-1] if n_elig > 0 else np.nan,
        'avg_enrolled': enrol_vals.mean() if n_enrol > 0 else np.nan,
        'sd_enrolled': enrol_vals.std() if n_enrol > 1 else np.nan,
        'min_enrolled': enrol_vals.min() if n_enrol > 0 else np.nan,
        'max_enrolled': enrol_vals.max() if n_enrol > 0 else np.nan,
        'first_enrolled': enrol_vals.iloc[0] if n_enrol > 0 else np.nan,
        'last_enrolled': enrol_vals.iloc[-1] if n_enrol > 0 else np.nan,
        'ssa': group['ssa'].iloc[-1]
    })

pen_2013 = ma_penetration.groupby(['fips', 'state', 'county', 'year']).apply(agg_penetration).reset_index()
print(f"Penetration data shape: {pen_2013.shape}")

Penetration data shape: (3224, 19)


/tmp/ipykernel_1182301/359218201.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pen_2013 = ma_penetration.groupby(['fips', 'state', 'county', 'year']).apply(agg_penetration).reset_index()


### Star Rating

In [6]:
#Star Rating

def parse_number_like_readr(x):
    """
    Rough equivalent of readr::parse_number(as.character(.)):
    - extracts the first numeric token, ignoring commas, %, stray text.
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return float("nan")
    s = str(x).strip()
    if s == "":
        return float("nan")
    s = s.replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else float("nan")


ma_path_a = (
    "../ma-data/ma/star-ratings/Extracted Star Ratings/Part C 2013 Fall/"
    "2013_Part_C_Report_Card_Master_Table_2012_10_17_Star.csv"
)

rating_vars_2013 = (
    ["contractid", "org_type", "org_marketing", "contract_name", "org_parent"]
    + [f"metric_{i:02d}" for i in range(1, 34)]

)
star_data_a = pd.read_csv(
    ma_path_a,
    skiprows=4,
    names=rating_vars_2013,        
    header=None,
    na_values=["", "NA", "*"],
    keep_default_na=True,
    encoding = "latin-1"
)

exclude_cols = {"contractid", "org_type", "contract_name", "org_marketing", "org_parent"}
cols_to_parse = [c for c in star_data_a.columns if c not in exclude_cols]
for c in cols_to_parse:
    star_data_a[c] = star_data_a[c].map(parse_number_like_readr).astype("float64")


ma_path_b = (
    "../ma-data/ma/star-ratings/Extracted Star Ratings/Part C 2013 Fall/"
    "2013_Part_C_Report_Card_Master_Table_2012_10_17_Summary.csv"
)
star_data_b = pd.read_csv(
    ma_path_b,
    skiprows=2,
    names=[
        "contractid","org_type","org_marketing","contract_name","org_parent",
        "partc_score","partc_lowscore","partc_highscore",
        "partcd_score","partcd_lowscore","partcd_highscore"
    ],
    header=None,
    na_values=["", "NA", "*"],
    keep_default_na=True,
    encoding = "latin-1"
)

star_data_b = star_data_b.assign(
    new_contract=lambda d: (
        (d["partc_score"] == "Plan too new to be measured")
        | (d["partcd_score"] == "Plan too new to be measured")
    ).astype("int64")
)

star_data_b["partc_score"] = star_data_b.apply(
    lambda r: float("nan") if r["new_contract"] == 1 else parse_number_like_readr(r["partc_score"]),
    axis=1
).astype("float64")

star_data_b["partcd_score"] = star_data_b.apply(
    lambda r: float("nan") if r["new_contract"] == 1 else parse_number_like_readr(r["partcd_score"]),
    axis=1
).astype("float64")

star_data_b["low_score"] = (star_data_b["partc_lowscore"] == "Yes").astype("float64")

star_data_b = star_data_b.loc[:, ["contractid", "new_contract", "low_score", "partc_score", "partcd_score"]]

final_star_ratings = (
    star_data_a
    .drop(columns=["contract_name", "org_type", "org_marketing"], errors="ignore")
    .merge(star_data_b, on="contractid", how="left") 
    .assign(year=2013)
)

### Rebate Data

In [7]:
def parse_number_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace(r"[^\d\.\-]+", "", regex=True)
    return pd.to_numeric(s, errors="coerce")

ma_path_a = "../ma-data/ma/cms-payment/2013/2013PartCPlan Level.xlsx"
risk_rebate_a = pd.read_excel(
    ma_path_a,
    sheet_name=0,
    usecols="A:G",       # note: only A–G in 2013
    skiprows=3,
    nrows=2968 - 4 + 1,
    header=None,
    names=[
        "contractid","planid","contract_name","plan_type",
        "riskscore_partc","payment_partc","rebate_partc"
    ],
)

ma_path_b = "../ma-data/ma/cms-payment/2013/2013PartDPlans.xlsx"
risk_rebate_b = pd.read_excel(
    ma_path_b,
    sheet_name=0,
    usecols="A:H",
    skiprows=3,
    nrows=3836 - 4 + 1,
    header=None,
    names=[
        "contractid","planid","contract_name","plan_type",
        "directsubsidy_partd","riskscore_partd","reinsurance_partd","costsharing_partd"
    ],
)

for col in ["riskscore_partc", "payment_partc", "rebate_partc"]:
    risk_rebate_a[col] = parse_number_series(risk_rebate_a[col])

risk_rebate_a["planid"] = pd.to_numeric(risk_rebate_a["planid"], errors="coerce")
risk_rebate_a["year"] = 2013

risk_rebate_a = risk_rebate_a[
    ["contractid","planid","contract_name","plan_type",
     "riskscore_partc","payment_partc","rebate_partc","year"]
]

for col in ["directsubsidy_partd", "reinsurance_partd", "costsharing_partd"]:
    risk_rebate_b[col] = parse_number_series(risk_rebate_b[col])

risk_rebate_b["payment_partd"] = (
    risk_rebate_b["directsubsidy_partd"]
    + risk_rebate_b["reinsurance_partd"]
    + risk_rebate_b["costsharing_partd"]
)

risk_rebate_b["planid"] = pd.to_numeric(risk_rebate_b["planid"], errors="coerce")

risk_rebate_b = risk_rebate_b[
    ["contractid","planid","payment_partd",
     "directsubsidy_partd","reinsurance_partd","costsharing_partd",
     "riskscore_partd"]
]

rebate_2013 = risk_rebate_a.merge(
    risk_rebate_b,
    on=["contractid","planid"],
    how="left"
)
     

### Benchmark Data

In [11]:
#Benchmarks 2013

# Read CSV (skip first 4 rows, specify column names)
bench_data = pd.read_csv(
    "../ma-data/ma/benchmarks/ratebook2013/CountyRate2013.csv",
    skiprows=4,
    names=[
        "ssa", "state", "county_name", "risk_star5",
        "risk_star45", "risk_star4", "risk_star35",
        "risk_star3", "risk_star25", "esrd_ab"
    ]
)

# Select columns and add missing numeric columns
final_benchmark = (
    bench_data[
        ["ssa", "risk_star5", "risk_star45", "risk_star4",
         "risk_star35", "risk_star3", "risk_star25"]
    ]
    .assign(
        aged_parta=np.nan,
        aged_partb=np.nan,
        risk_ab=np.nan,
        risk_bonus5=np.nan,
        risk_bonus35=np.nan,
        risk_bonus0=np.nan,
        year=2013
    )
)

print(final_benchmark.head())

      ssa risk_star5 risk_star45 risk_star4 risk_star35 risk_star3  \
0  1000.0     830.14      822.15     822.15      818.15     814.15   
1  1010.0     859.84      844.20     844.20      836.39     828.57   
2  1020.0     790.47      783.18     783.18      779.54     775.89   
3  1030.0     892.49      876.27     876.27      868.15     860.04   
4  1040.0     873.57      857.68     857.68      849.74     841.80   

  risk_star25  aged_parta  aged_partb  risk_ab  risk_bonus5  risk_bonus35  \
0      790.16         NaN         NaN      NaN          NaN           NaN   
1      781.67         NaN         NaN      NaN          NaN           NaN   
2      754.03         NaN         NaN      NaN          NaN           NaN   
3      811.36         NaN         NaN      NaN          NaN           NaN   
4      794.15         NaN         NaN      NaN          NaN           NaN   

   risk_bonus0  year  
0          NaN  2013  
1          NaN  2013  
2          NaN  2013  
3          NaN  2013  
4

### FFS data

In [16]:
# FFS 2013
path = "../ma-data/ffs-costs/Extracted Data/Aged Only/aged13.csv"

ffs_data = pd.read_csv(
    path,
    skiprows=2,
    header=None,
    na_values=["*", "."]
)

ffs_data = ffs_data.iloc[:, :15].copy()

ffs_data.columns = [
    "ssa", "state", "county_name", "parta_enroll",
    "parta_reimb", "parta_percap", "parta_reimb_unadj",
    "parta_percap_unadj", "parta_ime", "parta_dsh",
    "parta_gme", "partb_enroll",
    "partb_reimb", "partb_percap",
    "mean_risk"
]


ffscosts_2013 = (
    ffs_data.loc[:, ["ssa", "state", "county_name",
                     "parta_enroll", "parta_reimb",
                     "partb_enroll", "partb_reimb",
                     "mean_risk"]]
    .assign(
        year=2013,
        ssa=lambda df: pd.to_numeric(df["ssa"], errors="coerce")
    )
)

cols_to_parse = ["parta_enroll", "parta_reimb", "partb_enroll", "partb_reimb", "mean_risk"]
for c in cols_to_parse:
    ffscosts_2013[c] = (
        ffscosts_2013[c]
        .astype(str)
        .str.replace(r"[^0-9\.\-]+", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA})
    )
    ffscosts_2013[c] = pd.to_numeric(ffscosts_2013[c], errors="coerce")


ae = ffscosts_2013["parta_enroll"].fillna(0)
be = ffscosts_2013["partb_enroll"].fillna(0)
ar = ffscosts_2013["parta_reimb"]
br = ffscosts_2013["partb_reimb"]

ffscosts_2013["avg_ffscost"] = np.where(
    (ae == 0) & (be == 0), 0,
    np.where(
        (ae == 0) & (be > 0), br / be,
        np.where(
            (ae > 0) & (be == 0), ar / ae,
            (ar / ae) + (br / be)
        )
    )
)

### Merging Data

In [17]:
# Start with plan data, inner join with service area
ma_2013 = plan_data_2013.merge(
    service_data_2013[['contractid', 'fips']],
    on=['contractid', 'fips'],
    how='inner'
)

# Filter out territories and certain plan types
excluded_states = ['VI', 'PR', 'MP', 'GU', 'AS', '']
ma_2013 = ma_2013[
    (~ma_2013['state'].isin(excluded_states)) &
    (ma_2013['snp'] == 'No') &
    ((ma_2013['planid'] < 800) | (ma_2013['planid'] >= 900)) &
    (ma_2013['planid'].notna()) &
    (ma_2013['fips'].notna())
]

# Prepare penetration data for join
pen_2013_join = pen_2013.copy()
pen_2013_join = pen_2013_join.rename(columns={'state': 'state_long', 'county': 'county_long'})
pen_2013_join['state_long'] = pen_2013_join['state_long'].str.lower()

# Keep only unique fips entries
pen_2013_join['ncount'] = pen_2013_join.groupby('fips')['fips'].transform('count')
pen_2013_join = pen_2013_join[pen_2013_join['ncount'] == 1].drop(columns=['ncount'])

# Join penetration data
ma_2013 = ma_2013.merge(pen_2013_join, on='fips', how='left', suffixes=('', '_pen'))

# Create state name lookup
state_2013 = ma_2013.groupby('state').apply(
    lambda g: g.loc[g['state_long'].notna(), 'state_long'].iloc[-1] if g['state_long'].notna().any() else None
).reset_index()
state_2013.columns = ['state', 'state_name']

# Join state names
full_2013 = ma_2013.merge(state_2013, on='state', how='left')

# Prepare landscape data for join
landscape_2013_join = landscape_2013.copy()
landscape_2013_join['state'] = landscape_2013_join['state'].str.lower()

# Join landscape data
full_2013 = full_2013.merge(
    landscape_2013_join,
    left_on=['contractid', 'planid', 'state_name', 'county'],
    right_on=['contractid', 'planid', 'state', 'county'],
    how='left',
    suffixes=('', '_land')
)

# Join rebate data (exclude contract_name and plan_type from rebate)
rebate_2013_join = rebate_2013.drop(columns=['contract_name', 'plan_type'], errors='ignore')
full_2013 = full_2013.merge(rebate_2013_join, on=['contractid', 'planid'], how='left', suffixes=('', '_reb'))

# Merge into ffs
full_2013 = full_2013.merge(
    ffscosts_2013[[
        "ssa",
        "year",
        "avg_ffscost",
        "parta_enroll",
        "partb_enroll"
    ]],
    on=["ssa", "year"],
    how="left"
)

# Merge Star rating

full_2013["contractid"] = full_2013["contractid"].astype(str).str.strip()
final_star_ratings["contractid"] = final_star_ratings["contractid"].astype(str).str.strip()

full_2013 = full_2013.merge(
    final_star_ratings,
    on=["contractid", "year"],
    how="left",
    suffixes=("", "_star")
)

#merge benchmarks
final_benchmark["ssa"] = pd.to_numeric(final_benchmark["ssa"], errors="coerce")
full_2013["ssa"] = pd.to_numeric(full_2013["ssa"], errors="coerce")

# Merge benchmark rates onto your main panel
full_2013 = full_2013.merge(
    final_benchmark,
    on=["ssa", "year"],
    how="left",
    suffixes=("", "_bench")
)


# Calculate basic_premium and bid
def calc_basic_premium(row):
    if row.get('rebate_partc', 0) > 0:
        return 0
    elif row.get('partd') == 'No' and pd.notna(row.get('premium')) and pd.isna(row.get('premium_partc')):
        return row.get('premium')
    else:
        return row.get('premium_partc')

def calc_bid(row):
    rebate = row.get('rebate_partc', 0) or 0
    basic_premium = row.get('basic_premium', 0) or 0
    payment_partc = row.get('payment_partc')
    riskscore_partc = row.get('riskscore_partc')
    
    if pd.isna(payment_partc) or pd.isna(riskscore_partc) or riskscore_partc == 0:
        return np.nan
    elif rebate == 0 and basic_premium > 0:
        return (payment_partc + basic_premium) / riskscore_partc
    elif rebate > 0 or basic_premium == 0:
        return payment_partc / riskscore_partc
    else:
        return np.nan

full_2013['basic_premium'] = full_2013.apply(calc_basic_premium, axis=1)
full_2013['bid'] = full_2013.apply(calc_bid, axis=1)

print(f"Final merged data shape: {full_2013.shape}")


/tmp/ipykernel_1182301/1631500508.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_2013 = ma_2013.groupby('state').apply(


Final merged data shape: (68117, 114)


In [18]:
# Save to CSV
output_path = "../data/output/data-2013-star.csv"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
full_2013.to_csv(output_path, index=False)
print(f"Data saved to {output_path}")

Data saved to ../data/output/data-2013-star.csv
